# Describe the request body

The `requestBody` field in OpenAPI describes the body of a request that an API client can send to the server, including the content type(s) supported and the schema for the body content.

When the endpoint handler method accepts parameters that are bound to the request body, ASP.NET Core generates a corresponding `requestBody` for the operation in the OpenAPI document. Metadata for the request body can also be specified using attributes or extension methods. Additional metadata can be set with a [document transformer] or [operation transformer].

If the endpoint does not define any parameters bound to the request body, but instead consumes the request body from the [`HttpContext`] directly, ASP.NET Core provides mechanisms to specify request body metadata. This is a common scenario for endpoints that process the request body as a stream.

Some request body metadata can be determined from the [`FromBody`] or [`FromForm`] parameters of the route handler method.

A description for the request body can be set with a [`[Description]`] attribute on the [`FromBody`] or [`FromForm`] parameter.

If the [`FromBody`] parameter is non-nullable and the `EmptyBodyBehavior` is not set to `EmptyBodyBehavior.Allow` in the [`FromBody`] attribute, the request body is required and the `required` field of the `requestBody` is set to `true` in the generated OpenAPI document.
Form bodies are always required and have `required` set to `true`.

Use a [document transformer] or an [operation transformer] to set the `example`, `examples`, or `encoding` fields, or to add specification extensions for the request body in the generated OpenAPI document.

Other mechanisms for setting request body metadata depend on the type of app being developed and are described in the following sections.

[document transformer]: https://learn.microsoft.com/aspnet/core/fundamentals/openapi/aspnetcore-openapi?view=aspnetcore-9.0#use-document-transformers
[operation transformer]: https://learn.microsoft.com/aspnet/core/fundamentals/openapi/aspnetcore-openapi?view=aspnetcore-9.0#use-operation-transformers

[`FromBody`]: https://docs.microsoft.com/dotnet/api/microsoft.aspnetcore.mvc.frombodyattribute
[`FromForm`]: https://docs.microsoft.com/dotnet/api/microsoft.aspnetcore.mvc.fromformattribute
[`HttpContext`]: https://docs.microsoft.com/dotnet/api/microsoft.aspnetcore.http.httpcontext
[`[Description]`]: https://learn.microsoft.com/dotnet/api/system.componentmodel.descriptionattribute


## Minimal APIs

The content types for the request body in the generated OpenAPI document are determined from the type of the parameter that is bound to the request body or specified with the [`Accepts`] extension method.
By default, the content type of a [`FromBody`] parameter will be `application/json` and the content type for [`FromForm`] parameter(s) will be `multipart/form-data` or `application/x-www-form-urlencoded`.

Support for these default content types is built in to Minimal APIs, but other content types require custom binding.
See the [Custom binding] topic of the Minimal APIs documentation for more information.

[Custom binding]: https://learn.microsoft.com/aspnet/core/fundamentals/minimal-apis/parameter-binding

There are several ways to specify a different content type for the request body.
If the type of the [`FromBody`] parameter implements [`IEndpointParameterMetadataProvider`], ASP.NET Core uses this interface to determine the content type(s) the request body. 
The framework uses the [`PopulateMetadata`] method of this interface to set the content type(s) and type of the body content of the request body. For example, a `Todo` class that accepts either `application/xml` or `text/xml` content-type can use [`IEndpointParameterMetadataProvider`] to provide this information to the framework.

```csharp
public class Todo : IEndpointParameterMetadataProvider
{
    public static void PopulateMetadata(ParameterInfo parameter, EndpointBuilder builder)
    {
        builder.Metadata.Add(new AcceptsMetadata(["application/xml", "text/xml"], typeof(XmlBody)));
    }
}
```

The [`Accepts`] extension method can also be used to specify the content type of the request body.
In the following example, the endpoint accepts a `Todo` object in the request body with an expected content-type of `application/xml`.

```csharp
app.MapPut("/todos/{id}", (int id, Todo todo) => ...)
  .Accepts<Todo>("application/xml");
```

Since `application/xml` is not a built-in content type, the `Todo` class must implement the [`IBindableFromHttpContext<T>`] interface to provide a custom binding for the request body. For example

```csharp
public class Todo : IBindableFromHttpContext<Todo>
{
    public static async ValueTask<Todo?> BindAsync(HttpContext context, ParameterInfo parameter)
    {
        var xmlDoc = await XDocument.LoadAsync(context.Request.Body, LoadOptions.None, context.RequestAborted);
        var serializer = new XmlSerializer(typeof(Todo));
        return (Todo?)serializer.Deserialize(xmlDoc.CreateReader());
    }
```

If the endpoint does not define any parameters bound to the request body, use the [`Accepts`] extension method to specify the content type that the endpoint accepts. 

Note that if you specify [`Accepts`] multiple times, only the last one will be used -- they are not combined.

In [ ]:
curl -s -D - -X POST `
  -H "Content-Type: application/json" `
  -d '{"text": "Hello, world!"}' `
  http://localhost:5110/optional-body

In [ ]:
curl -s -D - -X POST `
  http://localhost:5110/optional-body

In [ ]:
curl -s -D - -X POST `
  -H "Content-Type: application/json" `
  -d '{"text": "Hello, world!"}' `
  http://localhost:5110/allow-empty-body

In [ ]:
curl -s -D - -X POST `
  http://localhost:5110/allow-empty-body

## Controllers

In controller-based apps, the content type(s) for the request body in the generated OpenAPI document are determined from the type of the parameter that is bound to the request body, the [`InputFormatter`]s configured in the application, or by a [`[Consumes]`] attribute on the route handler method.

ASP.NET Core uses an [`InputFormatter`] to deserialize a [`FromBody`] request body.
InputFormatters are configured in the [`MvcOptions`] passed to the [`AddControllers`] extension method for the app's service collection.
Each input formatter declares the content types it can handle, in its [`SupportedMediaTypes`] property,
and the type(s) of body content it can handle, with its [`CanReadType`] method.

ASP.NET Core MVC includes built-in input formatters for JSON and XML, though only the JSON input formatter is enabled by default.
The built-in JSON input formatter supports the `application/json`, `text/json`, and `application/*+json` content types, and the built-in XML input formatter supports the `application/xml`, `text/xml`, and `application/*+xml` content types.

By default, the content type of a [`FromBody`] request body may be any content type accepted by an [`InputFormatter`] for the [`FromBody`] parameter type. For a request body with [`FromForm`] parameter(s) the default content types are `multipart/form-data` or `application/x-www-form-urlencoded`.These content types will be included in the generated OpenAPI document if the [`[Consumes]`] attribute is not specified on the route handler method.

The content type(s) accepted by a route handler can be restricted using a [filter] on the endpoint (action scope).
The [`[Consumes]`] attribute adds an action scope filter to the endpoint that restricts the content types that a route handler will accept.
In this case, the requestBody in the generated OpenAPI document will include only the content type(s) specified in the [`[Consumes]`] attribute.

[filter]: https://learn.microsoft.com/aspnet/core/mvc/controllers/filters

A [`[Consumes]`] attribute cannot add support for a content type that does not have an associated input formatter, and the generated OpenAPI document will not include any content types that do not have an associated input formatter.

For content types other than JSON or XML, you need to create a custom input formatter.
For more detailed information and examples, see [Custom formatters in ASP.NET Core Web API].

If the route handler does not have a [`FromBody`] or [`FromForm`] parameter, the route handler may read
the request body directly from the `Request.Body` stream and may use the [`[Consumes]`] attribute to
restrict the content types allowed, but no requestBody is generated in the OpenAPI document.

[Custom formatters in ASP.NET Core Web API]: https://learn.microsoft.com/aspnet/core/web-api/advanced/custom-formatters


In [ ]:
curl -s -D - -X POST `
  -H "Content-Type: application/x-www-form-urlencoded" `
  -d "name=Mr. Magoo" `
  http://localhost:5124/from-form

HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Date: Mon, 21 Oct 2024 14:08:51 GMT
Server: Kestrel
Transfer-Encoding: chunked

"Good to go - Name: Mr. Magoo"


In [ ]:
curl -s -D - -X POST `
  -H "Content-Type: multipart/form-data" `
  -F name=Mr.Magoo `
  http://localhost:5124/from-form

HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Date: Mon, 21 Oct 2024 14:12:33 GMT
Server: Kestrel
Transfer-Encoding: chunked

"Good to go - Name: Mr.Magoo"
